# US House Seat Allocation - 5-Member Proportional System

This notebook calculates how House seats would be distributed under a system where:
- Each state gets districts based on population (1.9M per district)
- Each district elects 5 members proportionally
- Seats are allocated using the largest remainder method with Droop quota (16.67%)

In [54]:
import pandas as pd
import math

## Data: 2020 Census Population by State

In [55]:
CENSUS_2020 = {
    'California': 39538223,
    'Texas': 29145505,
    'Florida': 21538187,
    'New York': 20201249,
    'Pennsylvania': 13002700,
    'Illinois': 12812508,
    'Ohio': 11799448,
    'Georgia': 10711908,
    'North Carolina': 10439388,
    'Michigan': 10077331,
    'New Jersey': 9288994,
    'Virginia': 8631393,
    'Washington': 7705281,
    'Arizona': 7151502,
    'Massachusetts': 7029917,
    'Tennessee': 6910840,
    'Indiana': 6785528,
    'Maryland': 6177224,
    'Missouri': 6154913,
    'Wisconsin': 5893718,
    'Colorado': 5773714,
    'Minnesota': 5706494,
    'South Carolina': 5118425,
    'Alabama': 5024279,
    'Louisiana': 4657757,
    'Kentucky': 4505836,
    'Oregon': 4237256,
    'Oklahoma': 3959353,
    'Connecticut': 3605944,
    'Utah': 3271616,
    'Iowa': 3190369,
    'Nevada': 3104614,
    'Arkansas': 3011524,
    'Mississippi': 2961279,
    'Kansas': 2937880,
    'New Mexico': 2117522,
    'Nebraska': 1961504,
    'Idaho': 1839106,
    'West Virginia': 1793716,
    'Hawaii': 1455271,
    'New Hampshire': 1377529,
    'Maine': 1362359,
    'Montana': 1084225,
    'Rhode Island': 1097379,
    'Delaware': 990837,
    'South Dakota': 886667,
    'North Dakota': 779094,
    'Alaska': 733391,
    'Vermont': 643077,
    'Wyoming': 576851,
}

## Data: 2020 Presidential Election Results by State

## Data: 2024 Presidential Election Results by State

In [56]:
ELECTION_2024 = {
    'California': {'Dem': 58.88, 'Rep': 38.59, 'Other': 2.53},
    'Texas': {'Dem': 42.41, 'Rep': 56.30, 'Other': 1.29},
    'Florida': {'Dem': 43.00, 'Rep': 56.14, 'Other': 0.86},
    'New York': {'Dem': 55.85, 'Rep': 43.90, 'Other': 0.25},
    'Pennsylvania': {'Dem': 48.64, 'Rep': 50.40, 'Other': 0.96},
    'Illinois': {'Dem': 54.80, 'Rep': 43.87, 'Other': 1.33},
    'Ohio': {'Dem': 43.89, 'Rep': 55.21, 'Other': 0.90},
    'Georgia': {'Dem': 48.47, 'Rep': 50.77, 'Other': 0.76},
    'North Carolina': {'Dem': 47.67, 'Rep': 51.09, 'Other': 1.24},
    'Michigan': {'Dem': 48.25, 'Rep': 49.68, 'Other': 2.07},
    'New Jersey': {'Dem': 51.46, 'Rep': 46.44, 'Other': 2.10},
    'Virginia': {'Dem': 51.65, 'Rep': 46.24, 'Other': 2.11},
    'Washington': {'Dem': 57.78, 'Rep': 38.81, 'Other': 3.41},
    'Arizona': {'Dem': 46.70, 'Rep': 52.14, 'Other': 1.16},
    'Massachusetts': {'Dem': 61.33, 'Rep': 36.48, 'Other': 2.19},
    'Tennessee': {'Dem': 34.41, 'Rep': 64.30, 'Other': 1.29},
    'Indiana': {'Dem': 39.74, 'Rep': 58.74, 'Other': 1.52},
    'Maryland': {'Dem': 63.05, 'Rep': 34.87, 'Other': 2.08},
    'Missouri': {'Dem': 40.09, 'Rep': 58.53, 'Other': 1.38},
    'Wisconsin': {'Dem': 48.78, 'Rep': 49.64, 'Other': 1.58},
    'Colorado': {'Dem': 54.82, 'Rep': 42.60, 'Other': 2.58},
    'Minnesota': {'Dem': 50.91, 'Rep': 46.82, 'Other': 2.27},
    'South Carolina': {'Dem': 40.33, 'Rep': 57.99, 'Other': 1.68},
    'Alabama': {'Dem': 34.15, 'Rep': 64.80, 'Other': 1.05},
    'Louisiana': {'Dem': 38.31, 'Rep': 60.01, 'Other': 1.68},
    'Kentucky': {'Dem': 33.91, 'Rep': 64.58, 'Other': 1.51},
    'Oregon': {'Dem': 54.58, 'Rep': 42.51, 'Other': 2.91},
    'Oklahoma': {'Dem': 31.82, 'Rep': 66.23, 'Other': 1.95},
    'Connecticut': {'Dem': 56.37, 'Rep': 41.55, 'Other': 2.08},
    'Utah': {'Dem': 39.35, 'Rep': 57.60, 'Other': 3.05},
    'Iowa': {'Dem': 42.71, 'Rep': 55.97, 'Other': 1.32},
    'Nevada': {'Dem': 47.18, 'Rep': 50.63, 'Other': 2.19},
    'Arkansas': {'Dem': 33.64, 'Rep': 63.94, 'Other': 2.42},
    'Mississippi': {'Dem': 37.59, 'Rep': 61.51, 'Other': 0.90},
    'Kansas': {'Dem': 41.28, 'Rep': 56.58, 'Other': 2.14},
    'New Mexico': {'Dem': 51.69, 'Rep': 46.16, 'Other': 2.15},
    'Nebraska': {'Dem': 38.83, 'Rep': 58.86, 'Other': 2.31},
    'Idaho': {'Dem': 30.23, 'Rep': 67.19, 'Other': 2.58},
    'West Virginia': {'Dem': 28.01, 'Rep': 70.48, 'Other': 1.51},
    'Hawaii': {'Dem': 60.62, 'Rep': 37.78, 'Other': 1.60},
    'New Hampshire': {'Dem': 50.64, 'Rep': 47.79, 'Other': 1.57},
    'Maine': {'Dem': 52.37, 'Rep': 45.14, 'Other': 2.49},
    'Montana': {'Dem': 38.64, 'Rep': 58.57, 'Other': 2.79},
    'Rhode Island': {'Dem': 55.48, 'Rep': 42.17, 'Other': 2.35},
    'Delaware': {'Dem': 56.58, 'Rep': 41.77, 'Other': 1.65},
    'South Dakota': {'Dem': 33.94, 'Rep': 63.44, 'Other': 2.62},
    'North Dakota': {'Dem': 30.94, 'Rep': 66.96, 'Other': 2.10},
    'Alaska': {'Dem': 42.82, 'Rep': 53.05, 'Other': 4.13},
    'Vermont': {'Dem': 63.61, 'Rep': 33.49, 'Other': 2.90},
    'Wyoming': {'Dem': 25.91, 'Rep': 71.94, 'Other': 2.15},
}

In [57]:
ELECTION_2020 = {
    'California': {'Dem': 63.48, 'Rep': 34.32, 'Other': 2.20},
    'Texas': {'Dem': 46.48, 'Rep': 52.06, 'Other': 1.46},
    'Florida': {'Dem': 47.86, 'Rep': 51.22, 'Other': 0.92},
    'New York': {'Dem': 60.86, 'Rep': 37.75, 'Other': 1.39},
    'Pennsylvania': {'Dem': 50.01, 'Rep': 48.84, 'Other': 1.15},
    'Illinois': {'Dem': 57.54, 'Rep': 40.55, 'Other': 1.91},
    'Ohio': {'Dem': 45.24, 'Rep': 53.27, 'Other': 1.49},
    'Georgia': {'Dem': 49.47, 'Rep': 49.24, 'Other': 1.29},
    'North Carolina': {'Dem': 48.59, 'Rep': 49.93, 'Other': 1.48},
    'Michigan': {'Dem': 50.62, 'Rep': 47.84, 'Other': 1.54},
    'New Jersey': {'Dem': 57.33, 'Rep': 41.35, 'Other': 1.32},
    'Virginia': {'Dem': 54.11, 'Rep': 44.00, 'Other': 1.89},
    'Washington': {'Dem': 57.97, 'Rep': 38.77, 'Other': 3.26},
    'Arizona': {'Dem': 49.36, 'Rep': 49.06, 'Other': 1.58},
    'Massachusetts': {'Dem': 65.60, 'Rep': 32.14, 'Other': 2.26},
    'Tennessee': {'Dem': 37.45, 'Rep': 60.66, 'Other': 1.89},
    'Indiana': {'Dem': 40.96, 'Rep': 57.02, 'Other': 2.02},
    'Maryland': {'Dem': 65.36, 'Rep': 32.15, 'Other': 2.49},
    'Missouri': {'Dem': 41.41, 'Rep': 56.80, 'Other': 1.79},
    'Wisconsin': {'Dem': 49.45, 'Rep': 48.82, 'Other': 1.73},
    'Colorado': {'Dem': 55.40, 'Rep': 41.90, 'Other': 2.70},
    'Minnesota': {'Dem': 52.40, 'Rep': 45.28, 'Other': 2.32},
    'South Carolina': {'Dem': 43.43, 'Rep': 55.11, 'Other': 1.46},
    'Alabama': {'Dem': 36.57, 'Rep': 62.03, 'Other': 1.40},
    'Louisiana': {'Dem': 39.85, 'Rep': 58.46, 'Other': 1.69},
    'Kentucky': {'Dem': 36.15, 'Rep': 62.09, 'Other': 1.76},
    'Oregon': {'Dem': 56.45, 'Rep': 40.37, 'Other': 3.18},
    'Oklahoma': {'Dem': 32.29, 'Rep': 65.37, 'Other': 2.34},
    'Connecticut': {'Dem': 59.26, 'Rep': 39.21, 'Other': 1.53},
    'Utah': {'Dem': 37.65, 'Rep': 58.13, 'Other': 4.22},
    'Iowa': {'Dem': 44.89, 'Rep': 53.09, 'Other': 2.02},
    'Nevada': {'Dem': 50.06, 'Rep': 47.67, 'Other': 2.27},
    'Arkansas': {'Dem': 34.78, 'Rep': 62.40, 'Other': 2.82},
    'Mississippi': {'Dem': 41.06, 'Rep': 57.60, 'Other': 1.34},
    'Kansas': {'Dem': 41.56, 'Rep': 56.21, 'Other': 2.23},
    'New Mexico': {'Dem': 54.29, 'Rep': 43.50, 'Other': 2.21},
    'Nebraska': {'Dem': 39.19, 'Rep': 58.22, 'Other': 2.59},
    'Idaho': {'Dem': 33.07, 'Rep': 63.84, 'Other': 3.09},
    'West Virginia': {'Dem': 29.69, 'Rep': 68.62, 'Other': 1.69},
    'Hawaii': {'Dem': 63.73, 'Rep': 34.27, 'Other': 2.00},
    'New Hampshire': {'Dem': 52.71, 'Rep': 45.36, 'Other': 1.93},
    'Maine': {'Dem': 53.09, 'Rep': 44.02, 'Other': 2.89},
    'Montana': {'Dem': 40.55, 'Rep': 56.92, 'Other': 2.53},
    'Rhode Island': {'Dem': 59.39, 'Rep': 38.61, 'Other': 2.00},
    'Delaware': {'Dem': 58.74, 'Rep': 39.77, 'Other': 1.49},
    'South Dakota': {'Dem': 35.61, 'Rep': 61.77, 'Other': 2.62},
    'North Dakota': {'Dem': 31.76, 'Rep': 65.11, 'Other': 3.13},
    'Alaska': {'Dem': 42.77, 'Rep': 52.83, 'Other': 4.40},
    'Vermont': {'Dem': 66.09, 'Rep': 30.67, 'Other': 3.24},
    'Wyoming': {'Dem': 26.55, 'Rep': 69.94, 'Other': 3.51},
}

## Functions

In [58]:
# Constants for seat allocation
TOTAL_US_POP_2020 = sum(CENSUS_2020.values())  # 330,760,625
TARGET_HOUSE_SEATS = 870  # Double the current House (435 * 2)
POP_PER_SEAT = TOTAL_US_POP_2020 / TARGET_HOUSE_SEATS  # ~380,185

def calculate_seats_and_districts(population):
    """
    Calculate total seats and district configuration for a state.
    
    "Double the House" rule: ~380,000 constituents per representative
    Each state gets seats proportional to population (minimum 1 seat).
    
    District Rules:
    - 5-seat and 3-seat districts for states with ≥5 seats
    - 4-seat STATEWIDE district for states with exactly 4 seats
    - 2-seat STATEWIDE district for very small states (2 total seats)
    - 3-seat STATEWIDE district for small states (3 total seats)
    
    2-seat and 4-seat districts are ONLY used as statewide at-large districts,
    never as sub-state districts.
    
    Returns:
        tuple: (total_seats, district_list)
        e.g., (13, [5, 5, 3]) for a state with 13 seats
    """
    # Calculate ideal seats based on population
    raw_seats = population / POP_PER_SEAT
    total_seats = max(1, round(raw_seats))
    
    # Configure districts based on total seats
    if total_seats == 1:
        # Single at-large representative (constitutional minimum)
        districts = [1]
    elif total_seats == 2:
        # Statewide 2-seat district (WY, VT, AK, ND, SD)
        districts = [2]
    elif total_seats == 3:
        # Statewide 3-seat district (DE, MT, RI)
        districts = [3]
    elif total_seats == 4:
        # Statewide 4-seat district (ME, NH, HI)
        # Cannot be made with 5+3, so use single statewide 4-seat
        districts = [4]
    else:
        # For ≥5 seats: use only 5-seat and 3-seat districts
        # Mathematical fact: any integer ≥5 except 7 can be made with 5s and 3s
        # And no state gets exactly 7 seats with our population/seat ratio
        districts = []
        remaining = total_seats
        
        # Find optimal combination of 5s and 3s
        # Strategy: maximize 5-seat districts, fill remainder with 3s
        num_5s = remaining // 5
        remainder = remaining % 5
        
        if remainder == 0:
            # Perfect fit with 5s only
            districts = [5] * num_5s
        elif remainder == 1:
            # 5k + 1: convert one 5 to two 3s → 5(k-1) + 3 + 3 = 5k + 1
            districts = [5] * (num_5s - 1) + [3, 3]
        elif remainder == 2:
            # 5k + 2: convert one 5 to two 3s, add one more 3? No.
            # Actually: 5k + 2 = 5(k-1) + 5 + 2 = 5(k-1) + 3 + 4? No.
            # Better: check if we can do it differently
            # 5k + 2: need to express as 5a + 3b where 5a + 3b = 5k + 2
            # If k >= 1: 5(k-1) + 3*2 + 1? No, that's 5k - 5 + 6 + 1 = 5k + 2. But +1 doesn't work.
            # Actually: 5*1 + 3*(-1) = 2, so we need k >= 2 to convert:
            # 5k + 2 = 5(k-2) + 5 + 5 + 2 = 5(k-2) + 3 + 3 + 3 + 3 = 5k - 10 + 12 = 5k + 2 ✓
            # So: reduce 5s by 2, add 4 threes
            if num_5s >= 2:
                districts = [5] * (num_5s - 2) + [3] * 4
            else:
                # Fallback for edge case (shouldn't happen with our data)
                districts = [5] * num_5s + [2]  # Only if absolutely needed
        elif remainder == 3:
            # 5k + 3: just add one 3-seat district
            districts = [5] * num_5s + [3]
        elif remainder == 4:
            # 5k + 4: convert one 5 to two 3s, add one more 3? 
            # 5(k-1) + 3 + 3 + 4? No.
            # 5k + 4 = 5(k-1) + 5 + 4 = 5(k-1) + 3 + 3 + 3 = 5k - 5 + 9 = 5k + 4 ✓
            districts = [5] * (num_5s - 1) + [3] * 3
        
        districts = sorted(districts, reverse=True)
    
    return total_seats, districts


def allocate_seats_multiparty_variable(party_shares, district_sizes):
    """
    Allocate seats using Largest Remainder Method for variable-size districts.
    
    Each district is allocated independently based on the same party vote shares,
    then seats are summed across all districts.
    
    Args:
        party_shares: dict with party names as keys and vote share as PERCENTAGES (0-100)
        district_sizes: list of district sizes, e.g., [5, 5, 3]
    
    Returns:
        dict with allocated seats for each party
    """
    total_allocation = {party: 0 for party in party_shares.keys()}
    
    for district_seats in district_sizes:
        # Allocate this district using Largest Remainder (Hare Quota)
        seats_allocated = {}
        remainders = {}
        
        total_allocated = 0
        for party, pct in party_shares.items():
            share = pct / 100.0
            ideal_seats = share * district_seats
            whole_seats = int(ideal_seats)
            seats_allocated[party] = whole_seats
            remainders[party] = ideal_seats - whole_seats
            total_allocated += whole_seats
        
        # Distribute remaining seats by largest remainder
        seats_remaining = district_seats - total_allocated
        sorted_parties = sorted(remainders.keys(), key=lambda p: remainders[p], reverse=True)
        
        for i in range(seats_remaining):
            party = sorted_parties[i]
            seats_allocated[party] += 1
        
        # Add to total
        for party, seats in seats_allocated.items():
            total_allocation[party] += seats
    
    return total_allocation


# Validation
print("Seat Allocation System: 'Double the House' Rule")
print("="*70)
print(f"Total US Population: {TOTAL_US_POP_2020:,}")
print(f"Target Seats: {TARGET_HOUSE_SEATS}")
print(f"Population per Seat: {POP_PER_SEAT:,.0f}")
print()
print("District Rules:")
print("  - 5-seat and 3-seat districts for states with ≥5 seats")
print("  - 4-seat STATEWIDE for states with 4 seats (ME, NH, HI)")
print("  - 2-seat STATEWIDE for very small states (WY, VT, AK, ND, SD)")
print()

# Test with all states to verify no 2-seat sub-districts
total_seats = 0
total_districts = 0
district_counts = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}

for state, pop in sorted(CENSUS_2020.items(), key=lambda x: x[1]):
    seats, districts = calculate_seats_and_districts(pop)
    total_seats += seats
    total_districts += len(districts)
    for d in districts:
        district_counts[d] = district_counts.get(d, 0) + 1
    
    district_str = '+'.join(str(d) for d in districts)
    if seats <= 5 or any(d == 2 for d in districts) or any(d == 4 for d in districts):
        print(f"{state:15s}: {seats:3d} seats [{district_str:15s}]")

print()
print(f"Total: {total_seats} seats in {total_districts} districts")
print(f"  5-seat districts: {district_counts.get(5, 0)}")
print(f"  4-seat districts: {district_counts.get(4, 0)} (statewide only)")
print(f"  3-seat districts: {district_counts.get(3, 0)}")
print(f"  2-seat districts: {district_counts.get(2, 0)} (statewide only)")
print(f"  1-seat districts: {district_counts.get(1, 0)}")

Seat Allocation System: 'Double the House' Rule
Total US Population: 330,760,625
Target Seats: 870
Population per Seat: 380,185

District Rules:
  - 5-seat and 3-seat districts for states with ≥5 seats
  - 4-seat STATEWIDE for states with 4 seats (ME, NH, HI)
  - 2-seat STATEWIDE for very small states (WY, VT, AK, ND, SD)

Wyoming        :   2 seats [2              ]
Vermont        :   2 seats [2              ]
Alaska         :   2 seats [2              ]
North Dakota   :   2 seats [2              ]
South Dakota   :   2 seats [2              ]
Delaware       :   3 seats [3              ]
Montana        :   3 seats [3              ]
Rhode Island   :   3 seats [3              ]
Maine          :   4 seats [4              ]
New Hampshire  :   4 seats [4              ]
Hawaii         :   4 seats [4              ]
West Virginia  :   5 seats [5              ]
Idaho          :   5 seats [5              ]
Nebraska       :   5 seats [5              ]

Total: 871 seats in 207 districts
  5-seat d

In [59]:
def allocate_seats_proportional(vote_percentages, districts, seats_per_district=5):
    """
    Allocate seats using the Largest Remainder Method (Droop Quota).
    
    Args:
        vote_percentages: dict with 'Dem', 'Rep', 'Other' percentages
        districts: number of districts in the state
        seats_per_district: seats per district (default 5)
    
    Returns:
        dict with allocated seats for each party
    """
    total_seats = districts * seats_per_district
    
    # Droop quota: 1/(seats_per_district + 1) = 1/6 = 16.67%
    quota_decimal = 1 / (seats_per_district + 1)
    quota_percent = quota_decimal * 100
    
    seats_allocated = {}
    remainders = {}
    
    # Step 1: Allocate safe seats based on quota
    total_allocated = 0
    for party, vote_pct in vote_percentages.items():
        # Number of quotas this party earned
        quotas_earned = vote_pct / quota_percent
        safe_seats = math.floor(quotas_earned) * districts
        seats_allocated[party] = safe_seats
        total_allocated += safe_seats
        
        # Store the fractional remainder for later
        remainders[party] = (quotas_earned - math.floor(quotas_earned)) * districts
    
    # Step 2: Allocate remaining seats based on largest remainders
    remaining_seats = total_seats - total_allocated
    
    # Sort parties by remainder (largest first)
    sorted_parties = sorted(remainders.items(), key=lambda x: x[1], reverse=True)
    
    for i in range(remaining_seats):
        if i < len(sorted_parties):
            party = sorted_parties[i][0]
            seats_allocated[party] += 1
    
    return seats_allocated


def allocate_seats_variable_districts(vote_percentages, district_sizes):
    """
    Allocate seats for 2-party (Dem/Rep/Other) using variable district sizes.
    Uses Largest Remainder Method for each district independently.
    
    Args:
        vote_percentages: dict with 'Dem', 'Rep', 'Other' percentages
        district_sizes: list of district sizes, e.g., [5, 5, 3]
    
    Returns:
        dict with allocated seats for each party
    """
    total_allocation = {party: 0 for party in vote_percentages.keys()}
    
    for district_seats in district_sizes:
        seats_allocated = {}
        remainders = {}
        total_allocated = 0
        
        for party, vote_pct in vote_percentages.items():
            share = vote_pct / 100.0
            ideal_seats = share * district_seats
            whole_seats = int(ideal_seats)
            seats_allocated[party] = whole_seats
            remainders[party] = ideal_seats - whole_seats
            total_allocated += whole_seats
        
        remaining_seats = district_seats - total_allocated
        sorted_parties = sorted(remainders.keys(), key=lambda p: remainders[p], reverse=True)
        
        for i in range(remaining_seats):
            party = sorted_parties[i]
            seats_allocated[party] += 1
        
        for party, seats in seats_allocated.items():
            total_allocation[party] += seats
    
    return total_allocation


def allocate_seats_multiparty(party_shares, districts, seats_per_district=5):
    """
    Allocate seats using the Largest Remainder Method (Hare Quota) for any number of parties.
    
    This is a generalized version that works with data-driven party clusters from CES analysis.
    
    Args:
        party_shares: dict with party names as keys and vote share as PERCENTAGES (0-100)
                     e.g., {'Progressives': 20.5, 'Conservatives': 17.8, ...}
        districts: number of districts in the state
        seats_per_district: seats per district (default 5)
    
    Returns:
        dict with allocated seats for each party
    """
    total_seats = districts * seats_per_district
    
    seats_allocated = {}
    remainders = {}
    
    # Step 1: Calculate each party's "ideal" seat share and allocate whole seats
    total_allocated = 0
    for party, pct in party_shares.items():
        # Convert percentage to decimal (e.g., 20.5% -> 0.205)
        share = pct / 100.0
        # Party's ideal number of seats based on their vote share
        ideal_seats = share * total_seats
        # Allocate the whole number portion
        whole_seats = int(ideal_seats)
        seats_allocated[party] = whole_seats
        total_allocated += whole_seats
        
        # Store the fractional remainder
        remainders[party] = ideal_seats - whole_seats
    
    # Step 2: Allocate remaining seats by largest remainder
    remaining_seats = total_seats - total_allocated
    
    # Sort parties by remainder (descending)
    sorted_by_remainder = sorted(remainders.items(), key=lambda x: x[1], reverse=True)
    
    for i in range(remaining_seats):
        if i < len(sorted_by_remainder):
            party = sorted_by_remainder[i][0]
            seats_allocated[party] += 1
    
    return seats_allocated

## Calculate Seat Allocations

In [60]:
results = []

for state in CENSUS_2020.keys():
    population = CENSUS_2020[state]
    votes = ELECTION_2020[state]
    
    # Calculate total seats and district configuration using new function
    total_seats, district_sizes = calculate_seats_and_districts(population)
    num_districts = len(district_sizes)
    
    # Allocate seats proportionally using variable district sizes
    seat_allocation = allocate_seats_variable_districts(votes, district_sizes)
    
    results.append({
        'State': state,
        'Population': population,
        'Districts': num_districts,
        'District_Config': '+'.join(str(d) for d in district_sizes),
        'Total_Seats': total_seats,
        'Dem_Seats': seat_allocation.get('Dem', 0),
        'Rep_Seats': seat_allocation.get('Rep', 0),
        'Other_Seats': seat_allocation.get('Other', 0),
        'Dem_Vote': votes['Dem'],
        'Rep_Vote': votes['Rep']
    })

df = pd.DataFrame(results)

# Calculate totals
totals = {
    'State': 'TOTAL',
    'Population': df['Population'].sum(),
    'Districts': df['Districts'].sum(),
    'District_Config': f"{df['Districts'].sum()} districts",
    'Total_Seats': df['Total_Seats'].sum(),
    'Dem_Seats': df['Dem_Seats'].sum(),
    'Rep_Seats': df['Rep_Seats'].sum(),
    'Other_Seats': df['Other_Seats'].sum(),
    'Dem_Vote': None,
    'Rep_Vote': None
}

df_with_totals = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)

print("2020 Election - Proportional Seat Allocation ('Double the House' Rule)")
print("="*100)
print(f"Total Seats: {int(totals['Total_Seats']):,}")
print(f"Total Districts: {int(totals['Districts']):,}")
print(f"\nDem Seats: {int(totals['Dem_Seats']):,} ({totals['Dem_Seats']/totals['Total_Seats']*100:.1f}%)")
print(f"Rep Seats: {int(totals['Rep_Seats']):,} ({totals['Rep_Seats']/totals['Total_Seats']*100:.1f}%)")
print(f"Other Seats: {int(totals['Other_Seats']):,} ({totals['Other_Seats']/totals['Total_Seats']*100:.1f}%)")
print()
df_with_totals

2020 Election - Proportional Seat Allocation ('Double the House' Rule)
Total Seats: 871
Total Districts: 207

Dem Seats: 449 (51.5%)
Rep Seats: 422 (48.5%)
Other Seats: 0 (0.0%)



/var/folders/41/p2v6mqss4hgbxb1dc92rghbr0000gn/T/ipykernel_2847/2593880245.py:43: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,State,Population,Districts,District_Config,Total_Seats,Dem_Seats,Rep_Seats,Other_Seats,Dem_Vote,Rep_Vote
0,California,39538223,22,5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+3+3+3,104,63,41,0,63.48,34.32
1,Texas,29145505,17,5+5+5+5+5+5+5+5+5+5+5+5+5+3+3+3+3,77,30,47,0,46.48,52.06
2,Florida,21538187,13,5+5+5+5+5+5+5+5+5+3+3+3+3,57,22,35,0,47.86,51.22
3,New York,20201249,11,5+5+5+5+5+5+5+5+5+5+3,53,32,21,0,60.86,37.75
4,Pennsylvania,13002700,8,5+5+5+5+5+3+3+3,34,21,13,0,50.01,48.84
5,Illinois,12812508,8,5+5+5+5+5+3+3+3,34,21,13,0,57.54,40.55
6,Ohio,11799448,7,5+5+5+5+5+3+3,31,12,19,0,45.24,53.27
7,Georgia,10711908,6,5+5+5+5+5+3,28,17,11,0,49.47,49.24
8,North Carolina,10439388,7,5+5+5+3+3+3+3,27,10,17,0,48.59,49.93
9,Michigan,10077331,7,5+5+5+3+3+3+3,27,17,10,0,50.62,47.84


## Results

In [61]:
# Calculate national totals
totals = {
    'State': 'NATIONAL TOTAL',
    'Population': df['Population'].sum(),
    'Districts': df['Districts'].sum(),
    'Total_Seats': df['Total_Seats'].sum(),
    'Dem_Seats': df['Dem_Seats'].sum(),
    'Rep_Seats': df['Rep_Seats'].sum(),
    'Other_Seats': df['Other_Seats'].sum(),
}

# Add totals row
df_with_totals = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)

# Display results
display_df = df_with_totals[['State', 'Districts', 'Total_Seats', 
                               'Dem_Seats', 'Rep_Seats', 'Other_Seats']].copy()

print("="*80)
print("US HOUSE SEAT ALLOCATION - 5-MEMBER PROPORTIONAL SYSTEM")
print("="*80)
print("\nSystem Parameters:")
print(f"  - District Size: 1,900,000 people")
print(f"  - Seats per District: 5")
print(f"  - Allocation Method: Largest Remainder (Droop Quota)")
print(f"  - Quota Threshold: {100/6:.2f}%")
print("\n")

display_df

US HOUSE SEAT ALLOCATION - 5-MEMBER PROPORTIONAL SYSTEM

System Parameters:
  - District Size: 1,900,000 people
  - Seats per District: 5
  - Allocation Method: Largest Remainder (Droop Quota)
  - Quota Threshold: 16.67%




,State,Districts,Total_Seats,Dem_Seats,Rep_Seats,Other_Seats
0,California,22,104,63,41,0
1,Texas,17,77,30,47,0
2,Florida,13,57,22,35,0
3,New York,11,53,32,21,0
4,Pennsylvania,8,34,21,13,0
5,Illinois,8,34,21,13,0
6,Ohio,7,31,12,19,0
7,Georgia,6,28,17,11,0
8,North Carolina,7,27,10,17,0
9,Michigan,7,27,17,10,0


## National Summary

## Calculate 2024 Seat Allocations

In [62]:
results_2024 = []

for state in CENSUS_2020.keys():
    population = CENSUS_2020[state]
    votes = ELECTION_2024[state]
    
    # Calculate total seats and district configuration using new function
    total_seats, district_sizes = calculate_seats_and_districts(population)
    num_districts = len(district_sizes)
    
    # Allocate seats proportionally using variable district sizes
    seat_allocation = allocate_seats_variable_districts(votes, district_sizes)
    
    results_2024.append({
        'State': state,
        'Population': population,
        'Districts': num_districts,
        'District_Config': '+'.join(str(d) for d in district_sizes),
        'Total_Seats': total_seats,
        'Dem_Seats': seat_allocation.get('Dem', 0),
        'Rep_Seats': seat_allocation.get('Rep', 0),
        'Other_Seats': seat_allocation.get('Other', 0),
        'Dem_Vote': votes['Dem'],
        'Rep_Vote': votes['Rep']
    })

df_2024 = pd.DataFrame(results_2024)

# Calculate totals
totals_2024 = {
    'State': 'TOTAL',
    'Population': df_2024['Population'].sum(),
    'Districts': df_2024['Districts'].sum(),
    'District_Config': f"{df_2024['Districts'].sum()} districts",
    'Total_Seats': df_2024['Total_Seats'].sum(),
    'Dem_Seats': df_2024['Dem_Seats'].sum(),
    'Rep_Seats': df_2024['Rep_Seats'].sum(),
    'Other_Seats': df_2024['Other_Seats'].sum(),
    'Dem_Vote': None,
    'Rep_Vote': None
}

df_2024_with_totals = pd.concat([df_2024, pd.DataFrame([totals_2024])], ignore_index=True)

print("2024 Election - Proportional Seat Allocation ('Double the House' Rule)")
print("="*100)
print(f"Total Seats: {int(totals_2024['Total_Seats']):,}")
print(f"Total Districts: {int(totals_2024['Districts']):,}")
print(f"\nDem Seats: {int(totals_2024['Dem_Seats']):,} ({totals_2024['Dem_Seats']/totals_2024['Total_Seats']*100:.1f}%)")
print(f"Rep Seats: {int(totals_2024['Rep_Seats']):,} ({totals_2024['Rep_Seats']/totals_2024['Total_Seats']*100:.1f}%)")
print(f"Other Seats: {int(totals_2024['Other_Seats']):,} ({totals_2024['Other_Seats']/totals_2024['Total_Seats']*100:.1f}%)")
print()
df_2024_with_totals

2024 Election - Proportional Seat Allocation ('Double the House' Rule)
Total Seats: 871
Total Districts: 207

Dem Seats: 415 (47.6%)
Rep Seats: 456 (52.4%)
Other Seats: 0 (0.0%)



/var/folders/41/p2v6mqss4hgbxb1dc92rghbr0000gn/T/ipykernel_2847/1893953247.py:43: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,State,Population,Districts,District_Config,Total_Seats,Dem_Seats,Rep_Seats,Other_Seats,Dem_Vote,Rep_Vote
0,California,39538223,22,5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+5+3+3+3,104,63,41,0,58.88,38.59
1,Texas,29145505,17,5+5+5+5+5+5+5+5+5+5+5+5+5+3+3+3+3,77,30,47,0,42.41,56.30
2,Florida,21538187,13,5+5+5+5+5+5+5+5+5+3+3+3+3,57,22,35,0,43.00,56.14
3,New York,20201249,11,5+5+5+5+5+5+5+5+5+5+3,53,32,21,0,55.85,43.90
4,Pennsylvania,13002700,8,5+5+5+5+5+3+3+3,34,13,21,0,48.64,50.40
5,Illinois,12812508,8,5+5+5+5+5+3+3+3,34,21,13,0,54.80,43.87
6,Ohio,11799448,7,5+5+5+5+5+3+3,31,12,19,0,43.89,55.21
7,Georgia,10711908,6,5+5+5+5+5+3,28,11,17,0,48.47,50.77
8,North Carolina,10439388,7,5+5+5+3+3+3+3,27,10,17,0,47.67,51.09
9,Michigan,10077331,7,5+5+5+3+3+3+3,27,10,17,0,48.25,49.68


## 2024 Results

In [63]:
# Calculate national totals
totals_2024 = {
    'State': 'NATIONAL TOTAL',
    'Population': df_2024['Population'].sum(),
    'Districts': df_2024['Districts'].sum(),
    'Total_Seats': df_2024['Total_Seats'].sum(),
    'Dem_Seats': df_2024['Dem_Seats'].sum(),
    'Rep_Seats': df_2024['Rep_Seats'].sum(),
    'Other_Seats': df_2024['Other_Seats'].sum(),
}

# Add totals row
df_2024_with_totals = pd.concat([df_2024, pd.DataFrame([totals_2024])], ignore_index=True)

# Display results
display_df_2024 = df_2024_with_totals[['State', 'Districts', 'Total_Seats', 
                                        'Dem_Seats', 'Rep_Seats', 'Other_Seats']].copy()

print("="*80)
print("2024 US HOUSE SEAT ALLOCATION - 5-MEMBER PROPORTIONAL SYSTEM")
print("="*80)
print("\nSystem Parameters:")
print(f"  - District Size: 1,900,000 people")
print(f"  - Seats per District: 5")
print(f"  - Allocation Method: Largest Remainder (Droop Quota)")
print(f"  - Quota Threshold: {100/6:.2f}%")
print("\n")

display_df_2024

2024 US HOUSE SEAT ALLOCATION - 5-MEMBER PROPORTIONAL SYSTEM

System Parameters:
  - District Size: 1,900,000 people
  - Seats per District: 5
  - Allocation Method: Largest Remainder (Droop Quota)
  - Quota Threshold: 16.67%




,State,Districts,Total_Seats,Dem_Seats,Rep_Seats,Other_Seats
0,California,22,104,63,41,0
1,Texas,17,77,30,47,0
2,Florida,13,57,22,35,0
3,New York,11,53,32,21,0
4,Pennsylvania,8,34,13,21,0
5,Illinois,8,34,21,13,0
6,Ohio,7,31,12,19,0
7,Georgia,6,28,11,17,0
8,North Carolina,7,27,10,17,0
9,Michigan,7,27,10,17,0


## 2024 National Summary

In [65]:
totals_row_2024 = df_2024_with_totals.iloc[-1]

print("2024 National Summary:")
print(f"  Total Districts: {int(totals_row_2024['Districts'])}")
print(f"  Total Seats: {int(totals_row_2024['Total_Seats'])}")
print(f"  Democratic Seats: {int(totals_row_2024['Dem_Seats'])} ({totals_row_2024['Dem_Seats']/totals_row_2024['Total_Seats']*100:.1f}%)")
print(f"  Republican Seats: {int(totals_row_2024['Rep_Seats'])} ({totals_row_2024['Rep_Seats']/totals_row_2024['Total_Seats']*100:.1f}%)")
print(f"  Other Seats: {int(totals_row_2024['Other_Seats'])} ({totals_row_2024['Other_Seats']/totals_row_2024['Total_Seats']*100:.1f}%)")

2024 National Summary:
  Total Districts: 207
  Total Seats: 871
  Democratic Seats: 415 (47.6%)
  Republican Seats: 456 (52.4%)
  Other Seats: 0 (0.0%)


## Comparison: 2020 vs 2024

In [66]:
# Create comparison DataFrame
totals_2020 = df_with_totals.iloc[-1]
totals_2024 = df_2024_with_totals.iloc[-1]

comparison = pd.DataFrame({
    'Election Year': ['2020', '2024', 'Change'],
    'Dem_Seats': [
        int(totals_2020['Dem_Seats']),
        int(totals_2024['Dem_Seats']),
        int(totals_2024['Dem_Seats']) - int(totals_2020['Dem_Seats'])
    ],
    'Rep_Seats': [
        int(totals_2020['Rep_Seats']),
        int(totals_2024['Rep_Seats']),
        int(totals_2024['Rep_Seats']) - int(totals_2020['Rep_Seats'])
    ],
    'Other_Seats': [
        int(totals_2020['Other_Seats']),
        int(totals_2024['Other_Seats']),
        int(totals_2024['Other_Seats']) - int(totals_2020['Other_Seats'])
    ],
    'Total_Seats': [
        int(totals_2020['Total_Seats']),
        int(totals_2024['Total_Seats']),
        int(totals_2024['Total_Seats']) - int(totals_2020['Total_Seats'])
    ]
})

print("="*80)
print("NATIONAL SEAT COMPARISON: 2020 vs 2024")
print("="*80)
print()
print(comparison.to_string(index=False))
print()
print("Percentage Change:")
print(f"  Democratic: {(totals_2024['Dem_Seats']/totals_2024['Total_Seats'] - totals_2020['Dem_Seats']/totals_2020['Total_Seats'])*100:+.2f} percentage points")
print(f"  Republican: {(totals_2024['Rep_Seats']/totals_2024['Total_Seats'] - totals_2020['Rep_Seats']/totals_2020['Total_Seats'])*100:+.2f} percentage points")
print(f"  Other: {(totals_2024['Other_Seats']/totals_2024['Total_Seats'] - totals_2020['Other_Seats']/totals_2020['Total_Seats'])*100:+.2f} percentage points")

NATIONAL SEAT COMPARISON: 2020 vs 2024

Election Year  Dem_Seats  Rep_Seats  Other_Seats  Total_Seats
         2020        449        422            0          871
         2024        415        456            0          871
       Change        -34         34            0            0

Percentage Change:
  Democratic: -3.90 percentage points
  Republican: +3.90 percentage points
  Other: +0.00 percentage points
